# Proyecto de Minería de Datos - Inicio CRISP-DM

Este notebook inicia el proceso CRISP-DM para el dataset ubicado en `archive`, consolidando **todos los CSV en un único DataFrame** para comenzar la fase de comprensión y preparación de datos.

**Nota:** no se ejecuta automáticamente; está listo para que continúes.

## Fases CRISP-DM (alcance de este notebook)
- 1) Comprensión del negocio (documentar objetivo/problema).
- 2) Comprensión de los datos (carga unificada + EDA inicial).
- 3) Preparación de los datos (siguientes pasos).
- 4) Modelado (pendiente).
- 5) Evaluación (pendiente).
- 6) Despliegue (no requerido, pero se dejará reflexión al final del proyecto).

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid", palette="viridis")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)

In [ ]:
# Ruta del dataset
DATA_DIR = Path("/home/diegono/VII/D.S./FIN/archive")
csv_files = sorted(DATA_DIR.glob("*.csv"))

print(f"CSV encontrados: {len(csv_files)}")
for f in csv_files:
    print("-", f.name)

In [ ]:
def read_csv_flexible(file_path: Path) -> pd.DataFrame:
    """Intenta cargar CSV con separadores comunes para evitar fallos de lectura."""
    separators = [",", ";", "\t", "|"]
    for sep in separators:
        try:
            df = pd.read_csv(file_path, sep=sep, engine="python")
            if df.shape[1] > 1:
                return df
        except Exception:
            continue
    return pd.read_csv(file_path)

# raw_tables mantiene cada CSV por separado, útil para análisis posteriores
raw_tables = {}

# frames se usa para crear el DataFrame unificado
frames = []

for file_path in csv_files:
    table_name = file_path.stem
    df_temp = read_csv_flexible(file_path)
    raw_tables[table_name] = df_temp.copy()

    df_with_source = df_temp.copy()
    df_with_source["source_file"] = file_path.name
    df_with_source["source_table"] = table_name
    frames.append(df_with_source)

print(f"Tablas cargadas en raw_tables: {list(raw_tables.keys())}")

In [ ]:
# DataFrame unificado con todos los CSV
df_all = pd.concat(frames, ignore_index=True, sort=False)

print("Forma del DataFrame unificado:", df_all.shape)
print("Número de columnas:", len(df_all.columns))

display(df_all.head())
display(df_all.sample(min(5, len(df_all)), random_state=42))

In [ ]:
# Resumen rápido de estructura y calidad de datos
quality_summary = pd.DataFrame({
    "dtype": df_all.dtypes.astype(str),
    "missing_count": df_all.isna().sum(),
    "missing_pct": (df_all.isna().mean() * 100).round(2),
    "n_unique": df_all.nunique(dropna=True)
}).sort_values("missing_pct", ascending=False)

display(quality_summary.head(30))

In [ ]:
# Visualización 1: registros por archivo fuente
plt.figure(figsize=(10, 4))
order_sources = df_all["source_table"].value_counts().index
sns.countplot(data=df_all, x="source_table", order=order_sources)
plt.title("Cantidad de registros por tabla fuente")
plt.xlabel("Tabla fuente")
plt.ylabel("N° registros")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
# Visualización 2: distribución de variables numéricas (primeras 8)
numeric_cols = df_all.select_dtypes(include="number").columns.tolist()
plot_cols = numeric_cols[:8]

if len(plot_cols) > 0:
    df_all[plot_cols].hist(figsize=(14, 8), bins=30, edgecolor="black")
    plt.suptitle("Distribuciones iniciales de variables numéricas", y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("No se detectaron columnas numéricas para histogramas.")

In [ ]:
# Visualización 3: correlación inicial en variables numéricas (hasta 12 columnas)
numeric_valid = df_all.select_dtypes(include="number").copy()
numeric_valid = numeric_valid.dropna(axis=1, how="all")
numeric_valid = numeric_valid.loc[:, numeric_valid.nunique(dropna=True) > 1]
corr_cols = numeric_valid.columns[:12]

if len(corr_cols) >= 2:
    corr_matrix = df_all[corr_cols].corr(numeric_only=True)
    plt.figure(figsize=(10, 8))
    sns.heatmap(corr_matrix, cmap="coolwarm", center=0, annot=False)
    plt.title("Mapa de correlación inicial (subset numérico)")
    plt.tight_layout()
    plt.show()
else:
    print("No hay suficientes columnas numéricas válidas para correlación.")

In [ ]:
# (Opcional) Guardar dataset unificado para usarlo en otros notebooks/scripts
output_path = Path("/home/diegono/VII/D.S./FIN/dataset_unificado.csv")
# df_all.to_csv(output_path, index=False)
print(f"Cuando quieras exportarlo, descomenta la línea to_csv. Ruta objetivo: {output_path}")

## Próximo paso recomendado (CRISP-DM)
Define el problema analítico (clasificación, regresión o clustering), selecciona variable objetivo (si aplica), y decide qué subset de `df_all` usarás para modelado.